In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd 

In [3]:
s1 = pd.read_csv("../student_resource/dataset/train/train_source1.tsv", sep='\t')
s2 = pd.read_csv("../student_resource/dataset/train/train_source2.tsv", sep='\t')
s3 = pd.read_csv("../student_resource/dataset/train/train_source3.tsv", sep='\t')
g = pd.read_csv("../student_resource/dataset/train/train_ground_truth.tsv", sep='\t')

## Normalization

In [4]:
import sys
sys.path.append("..")

from src.normalization import normalize_pipeline

In [5]:
s1_normalized = normalize_pipeline(s1)
s2_normalized = normalize_pipeline(s2)
s3_normalized = normalize_pipeline(s3)

In [6]:
s1_normalized.head()

,entity_id,business_name,business_address,country,clean_name,clean_address,geo_block_key,phonetic_hash,search_document
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,1795 westchester drive high point nc,us_high_point_nc,ORL S BRBRXP,orelee s barbershop 1795 westchester drive hig...
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,17560 ellis rd tahlequah ok,us_tahlequah_ok,PRM MN,prime money 17560 ellis rd tahlequah ok
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail inc,1712 montebello ave phoenix az,us_phoenix_az,B RTL INK,b retail inc 1712 montebello ave phoenix az
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US,christ chapel,2100 cameron drive unit apartment g dundalk md,us_dundalk_md,XRST XPL,christ chapel 2100 cameron drive unit apartmen...
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India,prabhav business center,797 lake town block a kolkata howrah west bengal,india_howrah_west_bengal_p,PRBHF BSNS SNTR,prabhav business center 797 lake town block a ...


In [7]:
s1_normalized['geo_block_key'].value_counts()

geo_block_key
us_unknown                 335415
india_unknown              108994
india_andhra_pradesh        14639
india_gurgaon_haryana       14530
us_wv                       13815
                            ...  
india_kurnool_telangana        50
us_sutton_ma                   50
us_ada_oh                      50
us_woods_cross_city_ut         50
us_scotia_ny                   50
Name: count, Length: 4788, dtype: int64

In [8]:
s1_normalized['geo_block_key'].count()

np.int64(2206821)

In [9]:
from src.blocking import run_multi_index_generation

### Execute Global Search (Indices 1 & 2) Across All Sources

In [ ]:
from src.blocking import run_multi_index_generation, run_multi_source_geo_blocking

def generate_global_candidate_pool(s1, s2, s3, top_k=15):
    """Executes Index 1 & 2 across all pairs and standardizes columns."""
    print("="*40)
    print("STAGE 1-3: GLOBAL BM25 INDICES")
    print("="*40)
    
    pairs_12 = run_multi_index_generation(s1, s2, top_k)
    pairs_13 = run_multi_index_generation(s1, s3, top_k)
    pairs_23 = run_multi_index_generation(s2, s3, top_k)
    
    std_cols = ['entity_A', 'entity_B', 'bm25_text_score', 'bm25_phonetic_score']
    
    if not pairs_12.empty: pairs_12.columns = std_cols
    if not pairs_13.empty: pairs_13.columns = std_cols
    if not pairs_23.empty: pairs_23.columns = std_cols
    
    master_pool = pd.concat([pairs_12, pairs_13, pairs_23], ignore_index=True)
    return master_pool.drop_duplicates(subset=['entity_A', 'entity_B'])

global_candidates_df = generate_global_candidate_pool(s1_normalized, s2_normalized, s3_normalized, top_k=15)

STAGE 1-3: GLOBAL BM25 INDICES

[INDIA | TEXT] S1: 883188 | S2: 2017799
  -> Tokenizing S2 (Target) text...


Split strings:   0%|          | 0/2017799 [00:00<?, ?it/s]

  -> Building BM25 Search Engine...


BM25S Count Tokens:   0%|          | 0/2017799 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/2017799 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/50000 [00:00<?, ?it/s]

     ...Processed 200000/883188


Split strings:   0%|          | 0/50000 [00:00<?, ?it/s]

     ...Processed 250000/883188


Split strings:   0%|          | 0/50000 [00:00<?, ?it/s]

### Execute local search (index3) and merge

In [ ]:
# Run Local Geo Search (Index 3)
geo_candidates_df = run_multi_source_geo_blocking(s1_normalized, s2_normalized, s3_normalized, top_k=15)

# Merge All Nets Together
print("\nMerging Local Geo matches into Master Candidate Pool...")
final_master_pool = pd.merge(
    global_candidates_df, 
    geo_candidates_df, 
    on=['entity_A', 'entity_B'], 
    how='outer'
).fillna(0.0)

print(f"Final Candidate Pool Size: {len(final_master_pool)}")

### Generate the Leaderboard Submission

In [ ]:
def generate_baseline_submission(master_pool_df, s1_df, output_path="matching_results.tsv"):
    print("Generating baseline submission for the leaderboard...")
    
    # 1. Naive Thresholding 
    strong_matches = master_pool_df[
        (master_pool_df['bm25_text_score'] > 15.0) | 
        (master_pool_df['geo_ngram_score'] > 0.5)
    ].copy()
    
    # 2. Group by Source 1 ID and create the comma-separated list of S2/S3 matches
    grouped = strong_matches.groupby('entity_A')['entity_B'].apply(
        lambda x: ','.join(x.dropna().unique())
    ).reset_index()
    
    grouped = grouped.rename(columns={
        'entity_A': 'source1_entity_id', 
        'entity_B': 'matched_entity_ids'
    })
    
    # 3. Ensure ALL Source 1 entities are present in the final file
    submission_df = pd.DataFrame({'source1_entity_id': s1_df['entity_id']})
    submission_df = pd.merge(submission_df, grouped, on='source1_entity_id', how='left')
    
    # 4. Fill missing matches with an empty string
    submission_df['matched_entity_ids'] = submission_df['matched_entity_ids'].fillna("")
    
    # 5. Export as TSV
    submission_df.to_csv(output_path, sep='\t', index=False)
    
    print(f"Saved submission to: {output_path}")
    print(f"Total rows in submission: {len(submission_df)}")
    print(f"S1 entities with at least one match: {len(submission_df[submission_df['matched_entity_ids'] != ''])}")
    
    return submission_df

submission = generate_baseline_submission(
    final_master_pool, 
    s1_normalized, 
    output_path="matching_results.tsv"
)